# Notebook 05: Visualization Dashboard

This notebook brings together all results from the previous notebooks into a comprehensive set of visualizations:

1. **Allele frequency bar charts** — current vs. NFDS equilibrium target
2. **Compatibility network graph** — genotypes as nodes, compatible crosses as edges
3. **Convergence plot** — allele frequency variance over generations by strategy
4. **Ranked crosses table** — recommended crosses with impact scores
5. **Crossing outcome heatmap** — offspring diversity for each cross

The optimization aims to **accelerate** the natural convergence that NFDS already drives — not to override it. Random mating under NFDS will eventually reach equilibrium; strategic crossing gets there faster.

In [1]:
import sys
import itertools
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
from scipy.optimize import minimize

sns.set_style("whitegrid")

# Import all shared utilities
sys.path.insert(0, "../src")
from polyploid_utils import (
    canonical, allele_frequencies, form_gametes, is_compatible,
    cross, crossing_compatibility, sample_offspring, simulate_generation,
    distance_from_equilibrium, enumerate_compatible_crosses,
    compute_greedy_weights, compute_optimal_weights,
    identify_rare_alleles, get_mandatory_rare_crosses, select_elites,
)

print("All utilities loaded from polyploid_utils (with allele preservation).")

All utilities loaded from polyploid_utils (with allele preservation).


In [2]:
# ---------------------------------------------------------------------------
# Population and precomputation
# ---------------------------------------------------------------------------
# This cell reproduces the setup from Notebook 04 so this notebook is
# self-contained. It: (1) defines the skewed population, (2) enumerates
# all SI-compatible crosses, (3) runs scipy optimization, and (4) computes
# greedy weights. All results feed into the visualizations below.

allele_pool = list(range(1, 9))

population = [
    (1, 1, 2, 3),
    (1, 2, 2, 4),
    (1, 1, 3, 5),
    (2, 2, 3, 4),
    (1, 2, 5, 6),
    (1, 3, 4, 5),
    (2, 3, 3, 6),
    (1, 1, 2, 6),
    (1, 2, 4, 4),
    (2, 3, 5, 6),
    (1, 1, 4, 5),
    (1, 2, 3, 3),
    (2, 2, 5, 6),
    (1, 3, 4, 6),
    (1, 2, 2, 5),
    (1, 1, 3, 7),
    (2, 4, 5, 8),
    (1, 3, 6, 7),
    (2, 3, 4, 8),
    (1, 2, 7, 8),
]

# Step 1: Enumerate compatible crosses using helper
compatible_crosses, allele_effect_matrix = enumerate_compatible_crosses(population, allele_pool)
n_crosses = len(compatible_crosses)
n_pop = len(population)
target_freq = 1.0 / len(allele_pool)

# Step 2: Fitness function (standalone, used by downstream visualization cells)
def fitness_function(weights):
    w = np.abs(weights)
    w_sum = w.sum()
    if w_sum < 1e-12:
        return 1e6
    w = w / w_sum
    expected_freqs = w @ allele_effect_matrix
    return float(np.sum((expected_freqs - target_freq) ** 2 / target_freq))

# Step 3: Scipy optimization using helper
optimal_weights, result = compute_optimal_weights(
    compatible_crosses, allele_effect_matrix, allele_pool
)

# Step 4: Greedy weights using helper
greedy_weights = compute_greedy_weights(
    population, allele_pool, compatible_crosses, allele_effect_matrix
)

print(f"Setup complete: {n_pop} individuals, {len(allele_pool)} alleles, {n_crosses} compatible crosses")

Setup complete: 20 individuals, 8 alleles, 261 compatible crosses


## 1. Allele Frequency Distribution: Current vs. NFDS Equilibrium Target

In [ ]:
# Allele frequency bar chart: current population vs. NFDS equilibrium target.
# Deviation annotations above each pair show how far each allele is from target:
#   Red (+) = overrepresented allele (too common, NFDS will reduce it)
#   Green (-) = underrepresented allele (too rare, NFDS will boost it)
# The distance metrics in the corner quantify overall population-level imbalance.
freqs = allele_frequencies(population, allele_pool)
alleles = list(freqs.keys())
freq_values = list(freqs.values())

fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
x = np.arange(len(alleles))
width = 0.35

bars1 = ax.bar(x - width/2, freq_values, width, label="Current", color="steelblue", edgecolor="white")
bars2 = ax.bar(x + width/2, [target_freq] * len(alleles), width, label="NFDS target (1/n)",
               color="coral", alpha=0.7, edgecolor="white")

# Annotate each allele's deviation from the uniform target
for i, (curr, tgt) in enumerate(zip(freq_values, [target_freq] * len(alleles))):
    diff = curr - tgt
    color = "red" if diff > 0 else "green"
    ax.annotate(f"{diff:+.3f}", xy=(x[i], max(curr, tgt) + 0.005),
                ha="center", fontsize=8, color=color, fontweight="bold")

ax.set_xlabel("S-allele", fontsize=12)
ax.set_ylabel("Frequency", fontsize=12)
ax.set_title("Allele Frequency Distribution: Current vs. NFDS Equilibrium Target", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([f"S{a}" for a in alleles], fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, max(freq_values) * 1.3)

# Display all three distance metrics for a complete picture of disequilibrium
d = distance_from_equilibrium(population, allele_pool)
ax.text(0.98, 0.95,
        f"Variance: {d['variance']:.6f}\nChi-sq: {d['chi_squared']:.4f}\nKL div: {d['kl_divergence']:.4f}",
        transform=ax.transAxes, ha="right", va="top",
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8), fontsize=10)

plt.show()

**Key findings:**

- The most overrepresented alleles (S1, S2) are carried by many individuals and dominate the frequency distribution, while rarer alleles (S7, S8) sit well below the equilibrium target. This skew is typical of bottlenecked populations where founder effects inflate a few alleles at the expense of others.
- Under NFDS, overrepresented alleles are a *disadvantage* — their carriers are incompatible with more potential mates. Rare alleles confer a mating advantage, so the natural tendency is toward equalization, but in small populations this process is slow and stochastic.
- The gap between current and target frequencies defines the optimization objective: strategic crossing aims to close this gap faster than random mating alone, prioritizing crosses that boost the rarest alleles.

## 2. Compatibility Network Graph

Nodes represent **unique genotypes** in the population. Edges connect compatible crossing pairs. Edge thickness reflects the **optimal crossing weight** (thicker = higher priority).

In [ ]:
# Compatibility network: each node is a unique genotype in the population,
# each directed edge represents an SI-compatible cross. Edge thickness is
# proportional to the optimal crossing weight — thicker edges are higher-priority
# crosses for accelerating convergence to NFDS equilibrium.
# Isolated nodes (if any) would indicate genotypes with no compatible mates.

# Build directed graph from unique genotypes
unique_genotypes = list(set(population))
genotype_labels = {g: f"({','.join(str(a) for a in g)})" for g in unique_genotypes}

G = nx.DiGraph()
for g in unique_genotypes:
    G.add_node(genotype_labels[g])

# Aggregate optimal weights per genotype pair (multiple individuals may share
# the same genotype, so we sum their weights)
edge_weights = {}
for k in range(n_crosses):
    i, j, compat = compatible_crosses[k]
    ga = population[i]
    gb = population[j]
    key = (genotype_labels[ga], genotype_labels[gb])
    if key not in edge_weights:
        edge_weights[key] = 0.0
    edge_weights[key] += optimal_weights[k]

# Only draw edges with meaningful weight to keep the graph readable
for (src, dst), weight in edge_weights.items():
    if weight > 1e-4:
        G.add_edge(src, dst, weight=weight)

fig, ax = plt.subplots(figsize=(12, 10), layout="constrained")

pos = nx.spring_layout(G, seed=42, k=2)

# Draw nodes (genotypes) and labels
nx.draw_networkx_nodes(G, pos, node_size=800, node_color="lightblue",
                       edgecolors="steelblue", linewidths=2, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, font_weight="bold", ax=ax)

# Draw edges with width scaled by optimization weight
edges = G.edges(data=True)
if edges:
    weights = [d["weight"] * 50 for _, _, d in edges]  # Scale for visibility
    max_w = max(weights) if weights else 1
    nx.draw_networkx_edges(G, pos, width=[w / max_w * 3 + 0.5 for w in weights],
                           alpha=0.5, edge_color="gray",
                           connectionstyle="arc3,rad=0.1",
                           arrows=True, arrowsize=10, ax=ax)

ax.set_title("Crossing Compatibility Network\n(edge width = optimization priority)", fontsize=14)
ax.axis("off")

plt.show()

print(f"Nodes (unique genotypes): {G.number_of_nodes()}")
print(f"Edges (compatible crosses): {G.number_of_edges()}")

**Key findings:**

- The network is densely connected — most genotype pairs are SI-compatible, which is expected in a population with many distinct alleles. Genotypes carrying common alleles (S1, S2) tend to be incompatible with each other (since they share alleles), while genotypes with rare alleles act as high-connectivity hubs.
- The thickest edges (highest optimization weight) connect genotypes that carry complementary rare alleles — these are the crosses that most efficiently move the population toward NFDS equilibrium.
- From a practical standpoint, hub genotypes with many thick outgoing edges are the most valuable crossing partners. If any of these individuals were lost, crossing flexibility would decrease significantly.

## 3. Convergence Plot: Strategy Comparison

Each generation, adaptive strategies **recompute** their crossing plan from the current population — ensuring weights and indices stay valid as the population evolves. Random mating is the baseline (NFDS only).

In [ ]:
# Multi-generation convergence comparison across four crossing strategies.
# Left panel (linear scale): shows the absolute rate of variance decline.
# Middle panel (log scale): reveals differences in convergence rate.
# Right panel: shows allele loss over generations.
np.random.seed(123)
random.seed(123)

n_generations = 20
n_trials = 5

def run_strategy(pop, allele_pool, n_gen, strategy="random", n_trials=5,
                 opt_maxiter=1000, preserve_rare=False, rare_threshold=0.05,
                 elite_frac=0.1):
    """Run an adaptive strategy over multiple generations, averaging across stochastic trials."""
    all_variances = np.zeros((n_trials, n_gen + 1))
    all_extinct = np.zeros((n_trials, n_gen + 1))
    for trial in range(n_trials):
        current_pop = list(pop)
        for gen in range(n_gen + 1):
            d = distance_from_equilibrium(current_pop, allele_pool)
            all_variances[trial, gen] = d["variance"]
            all_extinct[trial, gen] = d.get("extinct_alleles", 0)
            if gen < n_gen:
                if strategy == "random":
                    current_pop = simulate_generation(
                        current_pop,
                        allele_pool=allele_pool if preserve_rare else None,
                        preserve_rare=preserve_rare,
                        rare_threshold=rare_threshold,
                        elite_frac=elite_frac,
                    )
                else:
                    cc, aem = enumerate_compatible_crosses(current_pop, allele_pool)
                    rare_idx = None
                    if preserve_rare:
                        freqs = allele_frequencies(current_pop, allele_pool)
                        sorted_alleles = sorted(allele_pool)
                        rare_idx = [i for i, a in enumerate(sorted_alleles)
                                    if 0 < freqs.get(a, 0) < rare_threshold]
                    if strategy == "greedy":
                        weights = compute_greedy_weights(current_pop, allele_pool, cc, aem)
                    elif strategy == "optimized":
                        weights, _ = compute_optimal_weights(
                            cc, aem, allele_pool, maxiter=opt_maxiter,
                            rare_allele_indices=rare_idx,
                        )
                    else:
                        raise ValueError(f"Unknown strategy: {strategy}")
                    plan = [
                        (cc[k][0], cc[k][1], weights[k])
                        for k in range(len(cc))
                        if weights[k] > 1e-6
                    ]
                    current_pop = simulate_generation(
                        current_pop, crossing_plan=plan,
                        allele_pool=allele_pool if preserve_rare else None,
                        preserve_rare=preserve_rare,
                        rare_threshold=rare_threshold,
                        elite_frac=elite_frac,
                    )
    return (all_variances.mean(axis=0), all_variances.std(axis=0),
            all_extinct.mean(axis=0), all_extinct.std(axis=0))

random_mean, random_std, random_ext, _ = run_strategy(
    population, allele_pool, n_generations, "random", n_trials)
greedy_mean, greedy_std, greedy_ext, _ = run_strategy(
    population, allele_pool, n_generations, "greedy", n_trials)
optimal_mean, optimal_std, optimal_ext, _ = run_strategy(
    population, allele_pool, n_generations, "optimized", n_trials)
preserved_mean, preserved_std, preserved_ext, _ = run_strategy(
    population, allele_pool, n_generations, "optimized", n_trials,
    preserve_rare=True, rare_threshold=0.05, elite_frac=0.1)

generations = np.arange(n_generations + 1)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5), layout="constrained")

# Linear scale
for mean, std, label, color, marker in [
    (random_mean, random_std, "Random (NFDS only)", "steelblue", "o"),
    (greedy_mean, greedy_std, "Greedy adaptive", "forestgreen", "s"),
    (optimal_mean, optimal_std, "Optimized adaptive", "crimson", "^"),
    (preserved_mean, preserved_std, "Opt + Preservation", "darkgreen", "D"),
]:
    ax1.plot(generations, mean, f"-{marker}", color=color, markersize=4, label=label)
    ax1.fill_between(generations, mean - std, mean + std, alpha=0.12, color=color)

ax1.axhline(y=0, color="black", linestyle="--", alpha=0.3)
ax1.set_xlabel("Generation")
ax1.set_ylabel("Frequency variance")
ax1.set_title("Convergence (linear scale)")
ax1.legend(fontsize=7)

# Log scale
for mean, std, label, color, marker in [
    (random_mean, random_std, "Random (NFDS only)", "steelblue", "o"),
    (greedy_mean, greedy_std, "Greedy adaptive", "forestgreen", "s"),
    (optimal_mean, optimal_std, "Optimized adaptive", "crimson", "^"),
    (preserved_mean, preserved_std, "Opt + Preservation", "darkgreen", "D"),
]:
    ax2.semilogy(generations, np.maximum(mean, 1e-8), f"-{marker}", color=color,
                 markersize=4, label=label)

ax2.set_xlabel("Generation")
ax2.set_ylabel("Frequency variance (log scale)")
ax2.set_title("Convergence (log scale)")
ax2.legend(fontsize=7)

# Allele loss
for ext, label, color, marker in [
    (random_ext, "Random", "steelblue", "o"),
    (greedy_ext, "Greedy", "forestgreen", "s"),
    (optimal_ext, "Optimized", "crimson", "^"),
    (preserved_ext, "Opt + Preservation", "darkgreen", "D"),
]:
    ax3.plot(generations, ext, f"-{marker}", color=color, markersize=4, label=label)

ax3.set_xlabel("Generation")
ax3.set_ylabel("Extinct alleles")
ax3.set_title("Allele Loss Over Generations")
ax3.legend(fontsize=7)

plt.show()

print("\nFinal variance by strategy:")
print(f"  Random (NFDS only):    {random_mean[-1]:.6f}")
print(f"  Greedy adaptive:       {greedy_mean[-1]:.6f}")
print(f"  Optimized adaptive:    {optimal_mean[-1]:.6f}")
print(f"  Opt + Preservation:    {preserved_mean[-1]:.6f}")
print(f"\nFinal extinct alleles:")
print(f"  Random: {random_ext[-1]:.1f}, Greedy: {greedy_ext[-1]:.1f}, "
      f"Optimized: {optimal_ext[-1]:.1f}, Preserved: {preserved_ext[-1]:.1f}")

**Key findings:**

- Both optimization-based strategies converge significantly faster than random mating, confirming that directed crossing can accelerate the natural NFDS process. The log-scale view reveals that the optimized strategy reaches near-equilibrium variance orders of magnitude faster.
- The preservation strategy (Opt + Preservation) converges slightly slower than pure optimization — this is the expected tradeoff. By protecting rare alleles through elitism and mandatory rare-allele crosses, it sacrifices some convergence speed to prevent irreversible allele loss.
- The allele loss panel is the critical differentiator: pure optimization can lose rare alleles in pursuit of variance minimization, while the preservation strategy maintains allele diversity. In small real populations, allele loss is permanent and far more damaging than slower convergence.

## 4. Ranked Table of Recommended Crosses

These crosses accelerate convergence toward NFDS equilibrium by boosting rare allele frequencies. Impact score measures how much a given cross moves offspring allele frequencies toward the uniform target.

In [6]:
# Ranked table of recommended crosses, sorted by impact score.
# Impact score = how much a cross moves offspring allele frequencies toward
# NFDS equilibrium compared to the current population's chi-squared distance.
#   Positive impact = offspring from this cross are closer to equilibrium
#   Higher impact = more beneficial cross for equalizing allele frequencies
# Practitioners should prioritize crosses at the top of this list.
cross_rows = []
for k in range(n_crosses):
    i, j, compat = compatible_crosses[k]
    expected_freqs = allele_effect_matrix[k]
    # Chi-squared of this cross's expected offspring frequencies vs. uniform target
    chi_sq = float(np.sum((expected_freqs - target_freq) ** 2 / target_freq))

    # Impact: how much better this cross is vs. the current population state.
    # Positive means the cross produces offspring closer to equilibrium.
    baseline_chi = d["chi_squared"]
    impact = baseline_chi - chi_sq

    cross_rows.append({
        "Rank": 0,
        "Maternal": str(population[i]),
        "Pollen donor": str(population[j]),
        "Compatibility": f"{compat:.0%}",
        "Offspring chi-sq": chi_sq,
        "Impact score": impact,
        "Optimal weight": optimal_weights[k],
    })

df_ranked = pd.DataFrame(cross_rows)
df_ranked = df_ranked.sort_values("Impact score", ascending=False).reset_index(drop=True)
df_ranked["Rank"] = range(1, len(df_ranked) + 1)

print("Top 20 Recommended Crosses (by impact toward NFDS equilibrium):")
print()
display_cols = ["Rank", "Maternal", "Pollen donor", "Compatibility", "Impact score", "Optimal weight"]
print(df_ranked[display_cols].head(20).to_string(index=False, float_format="{:.4f}".format))

Top 20 Recommended Crosses (by impact toward NFDS equilibrium):

 Rank     Maternal Pollen donor Compatibility  Impact score  Optimal weight
    1 (1, 3, 6, 7) (2, 4, 5, 8)          100%        0.3325          0.0138
    2 (2, 4, 5, 8) (1, 3, 6, 7)          100%        0.3325          0.0138
    3 (2, 3, 4, 8) (1, 3, 6, 7)           50%        0.1658          0.0121
    4 (2, 4, 5, 8) (1, 3, 4, 6)           50%        0.1658          0.0040
    5 (1, 3, 4, 5) (1, 2, 7, 8)           50%        0.1658          0.0119
    6 (1, 2, 5, 6) (2, 3, 4, 8)           50%        0.1658          0.0074
    7 (1, 3, 4, 6) (2, 4, 5, 8)           50%        0.1658          0.0063
    8 (2, 3, 5, 6) (1, 2, 7, 8)           50%        0.1658          0.0153
    9 (1, 2, 7, 8) (1, 3, 4, 5)           50%        0.1658          0.0106
   10 (1, 2, 7, 8) (2, 3, 5, 6)           50%        0.1658          0.0151
   11 (2, 3, 4, 8) (1, 2, 5, 6)           50%        0.1658          0.0056
   12 (1, 2, 7, 8) (1, 

**Key findings:**

- The highest-impact crosses pair genotypes carrying complementary rare alleles — crosses involving S7 and S8 carriers dominate the top of the list, because these alleles are furthest below the NFDS target frequency.
- Full compatibility (100%) crosses produce the most diverse offspring, but even low-compatibility crosses (16.7%) can have high impact if they uniquely introduce rare allele combinations.
- For practitioners, this ranked list directly translates to a crossing priority schedule: begin with the top crosses and work downward, adjusting as allele frequencies shift each generation.

## 5. Crossing Outcome Heatmap

For a subset of genotypes, show the number of unique offspring genotypes produced by each cross and the compatibility level.

In [ ]:
# Dual heatmaps for a subset of genotypes (first 12 unique genotypes).
# Left heatmap — SI Compatibility: fraction of pollen gametes accepted.
#   - Diagonal = 0 (self-incompatibility)
#   - Higher values = fewer shared alleles between parents
#   - Asymmetric because SI is evaluated against the maternal parent
# Right heatmap — Offspring Diversity: number of unique offspring genotypes.
#   - More diverse offspring = wider range of allele combinations in next generation
#   - Crosses with high compatibility tend to produce more diverse offspring
#   - Zero offspring count = fully incompatible cross (no pollen accepted)
unique_genos = sorted(set(population))[:12]
n_g = len(unique_genos)

compat_grid = np.zeros((n_g, n_g))
offspring_count_grid = np.zeros((n_g, n_g))

for i, ga in enumerate(unique_genos):
    for j, gb in enumerate(unique_genos):
        if i == j:
            continue
        compat_grid[i, j] = crossing_compatibility(ga, gb)
        outcomes = cross(ga, gb)
        offspring_count_grid[i, j] = len(outcomes)

labels = [f"({','.join(str(a) for a in g)})" for g in unique_genos]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), layout="constrained")

# Left: compatibility heatmap (0 to 1 scale)
sns.heatmap(compat_grid, annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels,
            cmap="YlOrRd", vmin=0, vmax=1, ax=ax1, cbar_kws={"label": "Compatibility"})
ax1.set_xlabel("Pollen donor")
ax1.set_ylabel("Maternal parent")
ax1.set_title("SI Compatibility")
ax1.tick_params(axis="x", rotation=45)
ax1.tick_params(axis="y", rotation=0)

# Right: offspring diversity heatmap (count of unique genotypes)
sns.heatmap(offspring_count_grid, annot=True, fmt=".0f", xticklabels=labels, yticklabels=labels,
            cmap="Blues", ax=ax2, cbar_kws={"label": "Unique offspring genotypes"})
ax2.set_xlabel("Pollen donor")
ax2.set_ylabel("Maternal parent")
ax2.set_title("Offspring Diversity (unique genotype count)")
ax2.tick_params(axis="x", rotation=45)
ax2.tick_params(axis="y", rotation=0)

plt.show()

**Key findings:**

- The compatibility heatmap confirms the SI diagonal: all self-crosses are zero (self-incompatibility), and crosses between genotypes sharing alleles have reduced compatibility. Pairs with no shared alleles achieve 100% compatibility and produce the most diverse offspring.
- Offspring diversity strongly correlates with compatibility — fully compatible crosses (1.00) can produce up to 36 unique offspring genotypes (6 maternal gametes x 6 paternal gametes), while low-compatibility crosses are limited to 6 (only one paternal gamete passes SI).
- For conservation crossing programs, this means prioritizing genetically distinct parents yields both higher reproductive success and greater genetic diversity in the next generation — a double benefit.